# 카카오 + VWorld 기반 대지 경계 기준 최소거리 분석

In [ ]:
# 최초 1회만 실행
# !pip install requests python-dotenv pandas folium shapely pyproj

In [1]:
import json
import math
import os

import folium
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

KAKAO_KEY = os.getenv("KAKAO_REST_API_KEY")
VWORLD_KEY = os.getenv("VWORLD_API_KEY")
VWORLD_DOMAIN = os.getenv("VWORLD_DOMAIN", "https://devprofessional.xyz")

KAKAO_BASE = "https://dapi.kakao.com/v2/local"
KAKAO_HEADERS = {"Authorization": f"KakaoAK {KAKAO_KEY}"}
VWORLD_URL = "https://api.vworld.kr/req/data"

print("카카오 키:", bool(KAKAO_KEY))
print("VWorld 키:", bool(VWORLD_KEY))
print("VWorld 도메인:", VWORLD_DOMAIN)

카카오 키: True
VWorld 키: True
VWorld 도메인: https://devprofessional.xyz


In [2]:
CATEGORY_GROUP = {
    "지하철역": "SW8", "학교": "SC4", "병원": "HP8", "대형마트": "MT1",
    "편의점": "CS2", "은행": "BK9", "공공기관": "PO3", "문화시설": "CT1",
    "어린이집·유치원": "PS3", "주차장": "PK6", "약국": "PM9", "카페": "CE7",
}


In [6]:
#주소를 좌표로 변환
def geocode(address):

    res = requests.get(f"{KAKAO_BASE}/search/address.json",
                       headers=KAKAO_HEADERS, params={"query": address, "size": 1})
    docs = res.json()["documents"]
    if not docs:
        return None

    d = docs[0]
    road = d.get("road_address") or {}
    return {"input": address, "lng": float(d["x"]), "lat": float(d["y"]),
            "road_address": road.get("address_name", ""),
            "address_name": d.get("address_name", "")}

#업종 코드로 주변 POI 를 모은다 (최대 45건)
def search_category(code, lng, lat, radius=1500, max_results=45):
    out = []
    for page in range(1, 4):
        res = requests.get(
            f"{KAKAO_BASE}/search/category.json",
            headers=KAKAO_HEADERS,
            params={"category_group_code": code, "x": lng, "y": lat,
                    "radius": radius, "page": page, "size": 15, "sort": "distance"},
        )
        body = res.json()
        out.extend(body["documents"])
        if body["meta"]["is_end"] or len(out) >= max_results:
            break

    return out[:max_results]

In [5]:
ADDRESS = "서울 용산구 남산공원길 105"

site = geocode(ADDRESS)
site

{'input': '서울 용산구 남산공원길 105',
 'lng': 126.987867837993,
 'lat': 37.5511225714939,
 'road_address': '서울 용산구 남산공원길 105',
 'address_name': '서울 용산구 남산공원길 105'}

In [8]:
site['input'], site['address_name'], site['lng'], site['lat']

('서울 용산구 남산공원길 105', '서울 용산구 남산공원길 105', 126.987867837993, 37.5511225714939)

# 2. VWorld API 활용 필지 경계 가져오기

`LP_PA_CBND_BUBUN`(연속지적도)에 `geomFilter=POINT(경도 위도)` 로 질의하면
그 점이 놓인 필지를 리턴한다.

**요청 형식**

```
https://api.vworld.kr/req/data
  ?service=data&request=GetFeature&data=LP_PA_CBND_BUBUN
  &key=<인증키>&domain=<등록 URL>
  &geomFilter=POINT(127.108 37.401)&crs=EPSG:4326&format=json&size=10
```